# PhishNet-Transformer - Step 1: Dataset Preparation

Dataset: PhiUSIIL Phishing URL Dataset (Kaggle)
https://www.kaggle.com/datasets/ndarvind/phiusiil-phishing-url-dataset

Goal here is just to get a clean, balanced train/val/test split ready before I start on the actual models. Downloaded the CSV and kept it in the same folder as this notebook.

Note: in this dataset label=1 actually means legitimate and label=0 means phishing, which is backwards from what I'd expect, so I flip it below to the normal convention (1 = phishing).

### Load the data

Just checking the file loaded properly and looking at the shape/columns before doing anything else.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

DATA_PATH = "PhiUSIIL_Phishing_URL_Dataset.csv"  # <-- update this if your downloaded filename differs

df = pd.read_csv(DATA_PATH)

print("Shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
df.head()

Shape (rows, columns): (235795, 55)

Column names:
['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,...,0,0,1,34,20,28,119,0,124,1
1,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,...,0,0,1,50,9,8,39,0,217,1
2,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,...,0,0,1,10,2,7,42,2,5,1
3,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,...,1,1,1,3,27,15,22,1,31,1
4,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,...,1,0,1,244,15,34,72,1,85,1


Got 235,795 rows and 55 columns, which matches what the dataset page says. Good, file loaded fine.

### Keep only URL and label

The dataset has a bunch of pre-built feature columns (URL length, IP flags etc.) but I don't need those right now - I'm building my own features later for the classic ML model, and the transformer just needs the raw text. So dropping everything except URL and label for now.

In [2]:
data = df[['URL', 'label']].copy()
print("New shape:", data.shape)
data.head()

New shape: (235795, 2)


,URL,label
0,https://www.southbankmosaics.com,1
1,https://www.uni-mainz.de,1
2,https://www.voicefmradio.co.uk,1
3,https://www.sfnmjournal.com,1
4,https://www.rewildingargentina.org,1


### Fix the label

Flipping the label since PhiUSIIL has it backwards (1=legit in the raw file). After this, 1 = phishing everywhere in my project, which is easier to keep track of.

In [3]:
data = data.rename(columns={'label': 'label_raw'})
data['label'] = 1 - data['label_raw']   # flip: now 1 = phishing, 0 = legitimate
data = data.drop(columns=['label_raw'])

print("Label counts after fixing convention (1 = phishing, 0 = legitimate):")
print(data['label'].value_counts())

Label counts after fixing convention (1 = phishing, 0 = legitimate):
label
0    134850
1    100945
Name: count, dtype: int64


Counts look right after the flip - matches the numbers on the dataset page (100,945 phishing / 134,850 legit).

### Clean up

Dropping any nulls and duplicate URLs. Mainly worried about duplicates ending up in both train and test - that would make my test accuracy look better than it actually is.

In [4]:
before = len(data)

data = data.dropna(subset=['URL', 'label'])
data = data.drop_duplicates(subset=['URL'])

after = len(data)
print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning:  {after}")
print(f"Dropped: {before - after} rows (nulls and/or duplicate URLs)")

Rows before cleaning: 235795
Rows after cleaning:  235370
Dropped: 425 rows (nulls and/or duplicate URLs)


### Balance the classes

Full dataset is a bit skewed (more legit than phishing) and also just really big. I only need a few thousand of each for this project, so sampling 6000 phishing + 6000 legit = 12000 total, balanced 50/50. Using a fixed random_state so it's reproducible if I rerun this.

In [5]:
N_PER_CLASS = 6000  # adjust down (e.g. 2000) if you want faster experimentation first

phishing = data[data['label'] == 1].sample(n=min(N_PER_CLASS, (data['label']==1).sum()), random_state=42)
legit    = data[data['label'] == 0].sample(n=min(N_PER_CLASS, (data['label']==0).sum()), random_state=42)

balanced = pd.concat([phishing, legit]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced dataset shape:", balanced.shape)
print(balanced['label'].value_counts())
balanced.head()

Balanced dataset shape: (12000, 2)
label
1    6000
0    6000
Name: count, dtype: int64


,URL,label
0,http://www.wodemo.com,1
1,https://www.radyofm60.com,0
2,http://www.majimoeleanallin30minutes.com,1
3,https://www.fortbendstar.com,0
4,http://www.gosvish.com,1


### Train / val / test split

70/15/15 split. Using stratify so each split keeps the same 50/50 ratio - otherwise I could randomly end up with a weird imbalance in my test set by chance.

Split sizes and ratios look correct - roughly 8400/1800/1800 and all close to 50/50.

In [6]:
# First split off 70% train, 30% temp
train, temp = train_test_split(
    balanced, test_size=0.30, stratify=balanced['label'], random_state=42
)

# Then split temp evenly into validation (15%) and test (15%)
val, test = train_test_split(
    temp, test_size=0.50, stratify=temp['label'], random_state=42
)

print("Train:", train.shape)
print("Val:  ", val.shape)
print("Test: ", test.shape)

print("\nClass balance check (should all be close to 0.5 / 0.5):")
print("Train:\n", train['label'].value_counts(normalize=True))
print("Val:\n", val['label'].value_counts(normalize=True))
print("Test:\n", test['label'].value_counts(normalize=True))

Train: (8400, 2)
Val:   (1800, 2)
Test:  (1800, 2)

Class balance check (should all be close to 0.5 / 0.5):
Train:
 label
0    0.5
1    0.5
Name: proportion, dtype: float64
Val:
 label
0    0.5
1    0.5
Name: proportion, dtype: float64
Test:
 label
0    0.5
1    0.5
Name: proportion, dtype: float64


### Quick sanity check

Before moving on I just want to actually look at some real examples and make sure the labels make sense.

Phishing examples are mostly on free hosting platforms (webcindario, weeblysite, jimdosite) and a shortener link - makes sense, that's cheap infrastructure for throwaway phishing pages. Legit ones look like normal small websites. Labels look correct.

In [7]:
print("Sample PHISHING URLs (label = 1):")
print(train[train['label'] == 1]['URL'].head(5).to_string(index=False))

print("\nSample LEGITIMATE URLs (label = 0):")
print(train[train['label'] == 0]['URL'].head(5).to_string(index=False))

Sample PHISHING URLs (label = 1):
https://provi78arge.webcindario.com/
  https://att-106905.weeblysite.com/
https://tinyurl.com/blocca-pagamento
   http://www.karnalbreakingnews.com
       https://bisabs.jimdosite.com/

Sample LEGITIMATE URLs (label = 0):
      https://www.plasticfreejuly.org
        https://www.roseyleebooks.com
      https://www.rogueguitarshop.com
             https://www.auvernix.org
https://www.superdumbsupervillain.com


### Save the splits

Saving these so the next notebooks (classic ML baseline, then DistilBERT) use the exact same data - otherwise the comparison between models wouldn't be fair.

Step 1 done. Have train.csv / val.csv / test.csv saved, ready for the XGBoost baseline next.

In [8]:
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)

print("Saved train.csv, val.csv, test.csv to the current folder.")
print("\nFinal summary:")
print(f"  Train: {len(train)} rows")
print(f"  Val:   {len(val)} rows")
print(f"  Test:  {len(test)} rows")

Saved train.csv, val.csv, test.csv to the current folder.



Final summary:
  Train: 8400 rows
  Val:   1800 rows
  Test:  1800 rows


### Done

Clean, balanced, split dataset ready - 8400 train / 1800 val / 1800 test, saved as CSVs. Next up: classic ML baseline (XGBoost) on this same split.